In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
from langchain.text_splitter import Language
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser

repo_path = "/Users/efatsi/Desktop/Projects/franklin/django"

In [16]:
loader = GenericLoader.from_filesystem(
    repo_path,
    glob="**/*",
    suffixes=[".py"],
    exclude=["**/non-utf8-encoding.py"],
    parser=LanguageParser(language=Language.PYTHON, parser_threshold=500),
)
documents = loader.load()

In [17]:
len(documents)

1026

In [18]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=2000, chunk_overlap=200
)
texts = python_splitter.split_documents(documents)
len(texts)

1969

In [20]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

db = Chroma.from_documents(texts, OpenAIEmbeddings(disallowed_special=()))
retriever = db.as_retriever(
    search_type="mmr",  # Also test "similarity"
    search_kwargs={"k": 8},
)

In [21]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x10e73e710>, search_type='mmr', search_kwargs={'k': 8})

In [47]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationSummaryMemory
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model_name="gpt-4-1106-preview")
memory = ConversationSummaryMemory(
    llm=llm, memory_key="chat_history", return_messages=True
)
qa = ConversationalRetrievalChain.from_llm(llm, retriever=retriever, memory=memory)

In [ ]:
llm.invoke("tell me a knock knock joke")

In [48]:
qa.invoke("What are all the django relationships defined on the OneTimePayroll model in core/models.py?")

{'question': 'What are all the django relationships defined on the OneTimePayroll model in core/models.py?',
 'chat_history': [SystemMessage(content='')],
 'answer': 'In the `OneTimePayroll` model located within `core/models.py`, the following Django model relationships are defined:\n\n1. A `ForeignKey` relationship with the `Organization` model:\n   ```python\n   organization = models.ForeignKey(\n       Organization, on_delete=models.CASCADE, related_name="one_time_payrolls"\n   )\n   ```\n   This relationship links each `OneTimePayroll` instance to an `Organization`. The `on_delete=models.CASCADE` argument specifies that if the referenced `Organization` is deleted, the `OneTimePayroll` instance will also be deleted. The `related_name="one_time_payrolls"` argument allows the `Organization` instance to access its related `OneTimePayroll` instances with the attribute `one_time_payrolls`.\n\n2. A `GenericRelation` with the `OneTimePayment` model:\n   ```python\n   payments = GenericRela

In [33]:
qa.invoke("what about ZealRequest?")

{'question': 'what about ZealRequest?',
 'chat_history': [SystemMessage(content='The human asks the AI to list all of the model classes defined in the primary core/models.py file. The AI lists 30 model classes including OrganizationFeature, QuickbooksIntegration, Role, Paycheck, Notification, Report, and DeductionTemplate among others.')],
 'answer': 'The context does not provide enough information to determine whether the ZealRequest is included in the primary core/models.py file.'}